# Monitor załogi: trening modeli (Google Colab)

**Środowisko wykonawcze → Zmień typ → Python 3 + GPU L4** (High-RAM, jeśli dostępne).

Notebook tylko **trenuje** modele i zapisuje je jako pliki `.onnx` na Twoim Dysku Google
(`MyDrive/MonitorZalogi/modele`). Potem kopiujesz ten folder do `laptop/modele` i uruchamiasz `monitor.py`.

| # | Model | Zbiór danych | Wynik |
|---|-------|--------------|-------|
| 1 | Stan oczu (otwarte/zamknięte) → PERCLOS, mrugnięcia | MRL Eye | `oczy.onnx` |
| 2 | Emocje z twarzy (7 klas) | FER-2013 | `emocje_twarz.onnx` |
| 3 | Emocje z głosu (8 klas) | RAVDESS (audio) | `emocje_glos.onnx` |
| 4 | Próg ziewania (kalibracja) | YawDD | `konfig.json` |
| 5 | Poziom zmęczenia (czujny / obniżona czujność / senny) | UTA-RLDD | `zmeczenie.onnx` |
| 6 | Tętno z kamery (POS, bez treningu): walidacja | UBFC-rPPG | `rppg_wyniki.json` |

Sekcje 1–3 pobierają dane automatycznie. Dla 4–6 trzeba samemu pobrać dane (rejestracja / formularz)
i wrzucić je na Dysk do `MyDrive/MonitorZalogi/dane/<NAZWA>/` (zip albo rozpakowane).
Sekcja bez danych zostanie pominięta. Po przerwaniu sesji uruchom od góry: gotowe kroki są pomijane.

*PURE, COHFACE i UBFC-Phys (z PDF-a) można dodać później do walidacji rPPG i modelu stresu.*

In [ ]:
!pip -q install mediapipe onnx onnxruntime onnxscript kagglehub soundfile
import mediapipe, onnxruntime, torch
print('mediapipe', mediapipe.__version__, '| onnxruntime', onnxruntime.__version__, '| torch', torch.__version__)
# Jeśli pojawi się błąd importu (np. numpy), użyj: Środowisko wykonawcze -> Uruchom ponownie sesję, i odpal od tej komórki.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, math, glob, re, time, zipfile, shutil, pathlib, urllib.request
import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F

BAZA = pathlib.Path('/content/drive/MyDrive/MonitorZalogi')
DANE = BAZA / 'dane'      # tu wrzucasz zbiory pobierane ręcznie
MODELE = BAZA / 'modele'  # tu lądują wytrenowane modele (to kopiujesz na laptopa)
LOKALNE = pathlib.Path('/content/dane')  # szybki dysk lokalny Colaba
for p in (DANE, MODELE, LOKALNE):
    p.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'BRAK, zmień środowisko na GPU L4!')
print('CPU:', os.cpu_count())

KONFIG_PATH = MODELE / 'konfig.json'
KONFIG = json.loads(KONFIG_PATH.read_text()) if KONFIG_PATH.exists() else {}
def zapisz_konfig():
    KONFIG_PATH.write_text(json.dumps(KONFIG, indent=2, ensure_ascii=False))

Wspólny moduł cech: **ten sam plik** jest w `laptop/cechy.py`, więc cechy z treningu i z laptopa są identyczne.

In [ ]:
%%writefile /content/cechy.py
"""
Wspolny modul cech - uzywany IDENTYCZNIE w Colab (trening) i na laptopie (dzialanie).

Dzieki temu cechy liczone podczas treningu (np. na filmach UTA-RLDD) sa dokladnie
takie same jak cechy liczone na zywo z kamery laptopa.

Zawartosc:
  - FaceAnalyzer       : MediaPipe FaceLandmarker (478 punktow + blendshapes + poza glowy)
  - EAR / MAR          : wskazniki otwarcia oczu i ust
  - wycinki oczu/twarzy: wejscie do sieci oczu (MRL Eye) i emocji (FER-2013)
  - FatigueTracker     : okno 60 s -> PERCLOS, mrugniecia, ziewniecia, ruchy glowy
  - rPPG (POS)         : tetno z koloru skory twarzy (bez treningu)
  - audio              : przygotowanie 4-sekundowego okna glosu dla modelu wav2vec2
"""
from __future__ import annotations

import collections
import math
from dataclasses import dataclass, field

import cv2
import numpy as np

# --------------------------------------------------------------------------- stale
FACE_LANDMARKER_URL = (
    "https://storage.googleapis.com/mediapipe-models/face_landmarker/"
    "face_landmarker/float16/1/face_landmarker.task"
)

# Indeksy punktow siatki MediaPipe (478 pkt). Kolejnosc: p1..p6 jak w definicji EAR.
OKO_L = (33, 160, 158, 133, 153, 144)
OKO_P = (362, 385, 387, 263, 373, 380)
USTA_PION = ((82, 87), (13, 14), (312, 317))
USTA_POZIOM = (78, 308)
CZOLO = (109, 10, 338, 337, 151, 108)
POLICZEK_L = (117, 118, 101, 36, 205, 187, 123)
POLICZEK_P = (346, 347, 330, 266, 425, 411, 352)

ROZMIAR_OKA = 64        # wejscie sieci oczu (MRL Eye)
ROZMIAR_TWARZY = 48     # wejscie sieci emocji (FER-2013 ma 48x48)
MAKS_SZEROKOSC = 640    # klatki sa zmniejszane do tej szerokosci (trening i laptop)
TRACKER_FPS = 15        # z jaka czestotliwoscia karmimy FatigueTracker (trening == laptop)

AUDIO_SR = 16000
AUDIO_SEG = 4 * AUDIO_SR  # 4 s

# FER-2013: kolejnosc alfabetyczna folderow (tak jak ImageFolder / sorted())
EMOCJE_TWARZ = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
# RAVDESS: kody 01..08
EMOCJE_GLOS = ["neutral", "calm", "happy", "sad", "angry", "fearful", "disgust", "surprised"]
NEGATYWNE = {"angry", "disgust", "fear", "fearful", "sad"}
PL = {
    "angry": "zlosc", "disgust": "wstret", "fear": "strach", "fearful": "strach",
    "happy": "radosc", "neutral": "neutralny", "sad": "smutek", "surprise": "zaskoczenie",
    "surprised": "zaskoczenie", "calm": "spokoj",
}
ZMECZENIE_KLASY = ["czujny", "obnizona czujnosc", "senny"]  # UTA-RLDD: 0 / 5 / 10

FEATURE_NAMES = [
    "face_ratio", "perclos", "closed_mean", "blinks_per_min", "blink_dur_mean",
    "long_closures_per_min", "ear_mean", "ear_std", "mar_mean", "mar_p95",
    "yawns_per_min", "pitch_mean", "pitch_std", "yaw_std", "roll_std",
]


# --------------------------------------------------------------------------- narzedzia
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x = np.asarray(x, np.float64)
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


def resize_max(frame: np.ndarray, max_w: int = MAKS_SZEROKOSC) -> np.ndarray:
    h, w = frame.shape[:2]
    if w <= max_w:
        return frame
    s = max_w / w
    return cv2.resize(frame, (max_w, int(round(h * s))), interpolation=cv2.INTER_AREA)


def _d(a, b) -> float:
    return float(np.hypot(*(a - b)))


def ear(pts: np.ndarray, idx) -> float:
    p1, p2, p3, p4, p5, p6 = (pts[i] for i in idx)
    return (_d(p2, p6) + _d(p3, p5)) / (2.0 * _d(p1, p4) + 1e-6)


def ear_obu(pts: np.ndarray) -> float:
    return 0.5 * (ear(pts, OKO_L) + ear(pts, OKO_P))


def mar(pts: np.ndarray) -> float:
    v = np.mean([_d(pts[a], pts[b]) for a, b in USTA_PION])
    return float(v / (_d(pts[USTA_POZIOM[0]], pts[USTA_POZIOM[1]]) + 1e-6))


def poza_glowy(m) -> tuple[float, float, float]:
    """pitch, yaw, roll [stopnie] z macierzy transformacji MediaPipe (4x4)."""
    r = np.asarray(m, np.float64)[:3, :3]
    pitch = math.degrees(math.atan2(r[2, 1], r[2, 2]))
    yaw = math.degrees(math.asin(float(np.clip(-r[2, 0], -1.0, 1.0))))
    roll = math.degrees(math.atan2(r[1, 0], r[0, 0]))
    return pitch, yaw, roll


def wytnij_kwadrat(img: np.ndarray, cx: float, cy: float, bok: float, out: int) -> np.ndarray:
    """Kwadratowy wycinek (z powieleniem krawedzi poza obrazem) przeskalowany do out x out."""
    bok = max(int(round(bok)), 2)
    x0, y0 = int(round(cx - bok / 2)), int(round(cy - bok / 2))
    h, w = img.shape[:2]
    pl, pt = max(0, -x0), max(0, -y0)
    pr, pb = max(0, x0 + bok - w), max(0, y0 + bok - h)
    crop = img[max(0, y0):min(h, y0 + bok), max(0, x0):min(w, x0 + bok)]
    if crop.size == 0:
        return np.zeros((out, out), img.dtype)
    if pl or pt or pr or pb:
        crop = cv2.copyMakeBorder(crop, pt, pb, pl, pr, cv2.BORDER_REPLICATE)
    interp = cv2.INTER_AREA if bok > out else cv2.INTER_LINEAR
    return cv2.resize(crop, (out, out), interpolation=interp)


def wycinki_oczu(gray: np.ndarray, pts: np.ndarray, skala: float = 1.5) -> list[np.ndarray]:
    wyn = []
    for idx in (OKO_L, OKO_P):
        p = pts[list(idx)]
        c = p.mean(axis=0)
        bok = skala * _d(pts[idx[0]], pts[idx[3]])
        wyn.append(wytnij_kwadrat(gray, c[0], c[1], bok, ROZMIAR_OKA))
    return wyn


def wycinek_twarzy(gray: np.ndarray, pts: np.ndarray, margines: float = 0.10) -> np.ndarray:
    x0, y0 = pts.min(axis=0)
    x1, y1 = pts.max(axis=0)
    bok = max(x1 - x0, y1 - y0) * (1 + 2 * margines)
    return wytnij_kwadrat(gray, (x0 + x1) / 2, (y0 + y1) / 2, bok, ROZMIAR_TWARZY)


def srednie_rgb_skory(frame_bgr: np.ndarray, pts: np.ndarray) -> np.ndarray:
    """Srednie RGB z czola i policzkow (do rPPG)."""
    h, w = frame_bgr.shape[:2]
    maska = np.zeros((h, w), np.uint8)
    for poly in (CZOLO, POLICZEK_L, POLICZEK_P):
        cv2.fillPoly(maska, [pts[list(poly)].astype(np.int32)], 255)
    if cv2.countNonZero(maska) < 50:
        return np.full(3, np.nan)
    b, g, r, _ = cv2.mean(frame_bgr, mask=maska)
    return np.array([r, g, b], np.float64)


# --------------------------------------------------------------------------- MediaPipe
@dataclass
class Obserwacja:
    pts: np.ndarray
    ear: float
    mar: float
    pitch: float
    yaw: float
    roll: float
    blend: dict = field(default_factory=dict)
    rgb: np.ndarray | None = None


class FaceAnalyzer:
    """Opakowanie FaceLandmarkera MediaPipe (tryb VIDEO). Jedna instancja na proces."""

    def __init__(self, model_path: str):
        from mediapipe.tasks.python import BaseOptions, vision

        opts = vision.FaceLandmarkerOptions(
            base_options=BaseOptions(model_asset_path=str(model_path)),
            running_mode=vision.RunningMode.VIDEO,
            num_faces=1,
            output_face_blendshapes=True,
            output_facial_transformation_matrixes=True,
        )
        self._lm = vision.FaceLandmarker.create_from_options(opts)
        self._ts = -1

    def przetworz(self, frame_bgr: np.ndarray, t_sek: float, z_rgb: bool = True) -> Obserwacja | None:
        import mediapipe as mp

        ts = int(t_sek * 1000)
        if ts <= self._ts:
            ts = self._ts + 1
        self._ts = ts
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        res = self._lm.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), ts)
        if not res.face_landmarks:
            return None
        h, w = frame_bgr.shape[:2]
        pts = np.array([[p.x * w, p.y * h] for p in res.face_landmarks[0]], np.float32)
        blend = {c.category_name: float(c.score) for c in res.face_blendshapes[0]} if res.face_blendshapes else {}
        if res.facial_transformation_matrixes:
            pitch, yaw, roll = poza_glowy(res.facial_transformation_matrixes[0])
        else:
            pitch = yaw = roll = float("nan")
        return Obserwacja(
            pts=pts, ear=ear_obu(pts), mar=mar(pts), pitch=pitch, yaw=yaw, roll=roll, blend=blend,
            rgb=srednie_rgb_skory(frame_bgr, pts) if z_rgb else None,
        )

    def close(self):
        self._lm.close()


class OnnxModel:
    def __init__(self, path: str, watki: int | None = None):
        import onnxruntime as ort

        so = ort.SessionOptions()
        if watki:
            so.intra_op_num_threads = watki
            so.inter_op_num_threads = 1
        self.s = ort.InferenceSession(str(path), so, providers=["CPUExecutionProvider"])
        self.we = self.s.get_inputs()[0].name

    def __call__(self, x: np.ndarray) -> np.ndarray:
        return self.s.run(None, {self.we: np.ascontiguousarray(x, np.float32)})[0]


def prawd_zamkniecia(obs: Obserwacja, gray: np.ndarray, model_oczu: OnnxModel | None) -> float:
    """0..1 - na ile oczy sa zamkniete. Siec MRL Eye, a bez niej blendshape MediaPipe."""
    if model_oczu is not None:
        p = []
        for crop in wycinki_oczu(gray, obs.pts):
            logit = model_oczu(crop[None, None].astype(np.float32) / 255.0)
            p.append(softmax(logit)[0, 1])
        return float(np.mean(p))
    return 0.5 * (obs.blend.get("eyeBlinkLeft", 0.0) + obs.blend.get("eyeBlinkRight", 0.0))


def wejscie_twarzy(gray: np.ndarray, pts: np.ndarray) -> np.ndarray:
    return wycinek_twarzy(gray, pts)[None, None].astype(np.float32) / 255.0


# --------------------------------------------------------------------------- zmeczenie
def _segmenty(maska: np.ndarray, t: np.ndarray) -> list[float]:
    """Czasy trwania ciaglych odcinkow True."""
    dl, start = [], None
    for i, m in enumerate(maska):
        if m and start is None:
            start = i
        elif not m and start is not None:
            dl.append(t[i] - t[start])
            start = None
    if start is not None:
        dl.append(t[-1] - t[start])
    return dl


class FatigueTracker:
    """Okno przesuwne (domyslnie 60 s) -> wektor cech FEATURE_NAMES."""

    def __init__(self, okno: float = 60.0, prog_mar: float = 0.5):
        self.okno = okno
        self.prog_mar = prog_mar
        self.buf: collections.deque = collections.deque()

    def dodaj(self, t: float, obs: Obserwacja | None, zamk: float | None):
        nan = float("nan")
        if obs is None or zamk is None:
            self.buf.append((t, nan, nan, nan, nan, nan, nan))
        else:
            self.buf.append((t, zamk, obs.ear, obs.mar, obs.pitch, obs.yaw, obs.roll))
        while self.buf and self.buf[0][0] < t - self.okno:
            self.buf.popleft()

    def zakres(self) -> float:
        return self.buf[-1][0] - self.buf[0][0] if len(self.buf) > 1 else 0.0

    def cechy(self) -> np.ndarray:
        a = np.array(self.buf, np.float64)
        f = dict.fromkeys(FEATURE_NAMES, float("nan"))
        if len(a) < 2:
            return np.array([f[n] for n in FEATURE_NAMES], np.float32)
        t = a[:, 0]
        twarz = ~np.isnan(a[:, 1])
        dur = max(t[-1] - t[0], 1e-6)
        f["face_ratio"] = float(twarz.mean())
        if twarz.sum() >= 10:
            tf, b = t[twarz], a[twarz]
            zam = b[:, 1] > 0.5
            seg = _segmenty(zam, tf)
            mrug = [d for d in seg if 0.04 <= d < 0.5]
            dlugie = [d for d in seg if d >= 0.5]
            ziew = [d for d in _segmenty(b[:, 3] > self.prog_mar, tf) if d >= 1.0]
            f.update(
                perclos=float(zam.mean()),
                closed_mean=float(b[:, 1].mean()),
                blinks_per_min=len(mrug) / dur * 60,
                blink_dur_mean=float(np.mean(mrug)) if mrug else 0.0,
                long_closures_per_min=len(dlugie) / dur * 60,
                ear_mean=float(b[:, 2].mean()),
                ear_std=float(b[:, 2].std()),
                mar_mean=float(b[:, 3].mean()),
                mar_p95=float(np.percentile(b[:, 3], 95)),
                yawns_per_min=len(ziew) / dur * 60,
                pitch_mean=float(np.nanmean(b[:, 4])),
                pitch_std=float(np.nanstd(b[:, 4])),
                yaw_std=float(np.nanstd(b[:, 5])),
                roll_std=float(np.nanstd(b[:, 6])),
            )
        return np.array([f[n] for n in FEATURE_NAMES], np.float32)


def zmeczenie_heurystyka(c: np.ndarray) -> float:
    """Wynik 0-100, gdy nie ma wytrenowanego modelu zmeczenia."""
    f = dict(zip(FEATURE_NAMES, c))
    if not np.isfinite(f["perclos"]):
        return float("nan")
    s = f["perclos"] * 250 + f["long_closures_per_min"] * 10 + f["yawns_per_min"] * 15
    return float(np.clip(s, 0, 100))


# --------------------------------------------------------------------------- przetwarzanie filmow (Colab)
def klatki_filmu(path: str, fps_cel: float | None):
    """Generator (t_sek, klatka_bgr). fps_cel=None -> wszystkie klatki."""
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or fps < 1 or fps > 240:
        fps = 30.0
    krok = max(1, round(fps / fps_cel)) if fps_cel else 1
    i = 0
    while True:
        if i % krok == 0:
            ok, fr = cap.read()
        else:
            ok, fr = cap.grab(), None
        if not ok:
            break
        if fr is not None:
            yield i / fps, fr
        i += 1
    cap.release()


def cechy_z_filmu(path: str, landmarker: str, model_oczu: str | None, prog_mar: float,
                  okno: float = 60.0, krok: float = 10.0) -> np.ndarray:
    """Film -> macierz (n_okien, len(FEATURE_NAMES)). Uzywane do treningu na UTA-RLDD."""
    fa = FaceAnalyzer(landmarker)
    oczy = OnnxModel(model_oczu, watki=1) if model_oczu else None
    tr = FatigueTracker(okno, prog_mar)
    out, nastepny = [], okno
    for t, fr in klatki_filmu(path, TRACKER_FPS):
        fr = resize_max(fr)
        obs = fa.przetworz(fr, t, z_rgb=False)
        zamk = prawd_zamkniecia(obs, cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY), oczy) if obs else None
        tr.dodaj(t, obs, zamk)
        if t >= nastepny:
            out.append(tr.cechy())
            nastepny += krok
    fa.close()
    return np.array(out, np.float32).reshape(-1, len(FEATURE_NAMES))


def mar_z_filmu(path: str, landmarker: str) -> np.ndarray:
    """MAR klatka po klatce (TRACKER_FPS). Uzywane do kalibracji progu ziewania na YawDD."""
    fa = FaceAnalyzer(landmarker)
    wyn = []
    for t, fr in klatki_filmu(path, TRACKER_FPS):
        obs = fa.przetworz(resize_max(fr), t, z_rgb=False)
        if obs:
            wyn.append(obs.mar)
    fa.close()
    return np.array(wyn, np.float32)


def rgb_z_filmu(path: str, landmarker: str) -> tuple[np.ndarray, np.ndarray, float]:
    """(t, rgb[N,3], fps) dla kazdej klatki. Uzywane do walidacji rPPG na UBFC-rPPG."""
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    cap.release()
    fa = FaceAnalyzer(landmarker)
    t_l, rgb_l = [], []
    for t, fr in klatki_filmu(path, None):
        fr = resize_max(fr)
        obs = fa.przetworz(fr, t)
        t_l.append(t)
        rgb_l.append(obs.rgb if obs is not None else np.full(3, np.nan))
    fa.close()
    return np.array(t_l), np.array(rgb_l), float(fps)


# --------------------------------------------------------------------------- rPPG (POS)
def uzupelnij_nan(x: np.ndarray) -> np.ndarray:
    x = np.array(x, np.float64)
    for k in range(x.shape[1]):
        c = x[:, k]
        ok = np.isfinite(c)
        if ok.sum() >= 2 and not ok.all():
            c[~ok] = np.interp(np.flatnonzero(~ok), np.flatnonzero(ok), c[ok])
    return x


def przeprobkuj(t: np.ndarray, x: np.ndarray, fs: float) -> np.ndarray:
    tt = np.arange(t[0], t[-1], 1.0 / fs)
    return np.stack([np.interp(tt, t, x[:, k]) for k in range(x.shape[1])], axis=1)


def pos_bvp(rgb: np.ndarray, fs: float) -> np.ndarray:
    """Plane-Orthogonal-to-Skin (Wang i in., 2017)."""
    n = len(rgb)
    l = int(round(1.6 * fs))
    h = np.zeros(n)
    proj = np.array([[0.0, 1.0, -1.0], [-2.0, 1.0, 1.0]])
    for k in range(l, n + 1):
        c = rgb[k - l:k].T
        m = c.mean(axis=1, keepdims=True)
        if np.any(m <= 0):
            continue
        s = proj @ (c / m)
        hh = s[0] + (s[0].std() / (s[1].std() + 1e-9)) * s[1]
        h[k - l:k] += hh - hh.mean()
    return h


def pasmowy(x: np.ndarray, fs: float, lo: float = 0.7, hi: float = 3.0) -> np.ndarray:
    from scipy.signal import butter, filtfilt

    b, a = butter(3, [lo, hi], btype="band", fs=fs)
    return filtfilt(b, a, x)


def tetno_z_sygnalu(x: np.ndarray, fs: float, lo: float = 0.7, hi: float = 3.0) -> tuple[float, float]:
    """(BPM, SNR w dB) z widma sygnalu."""
    x = (x - x.mean()) * np.hanning(len(x))
    nfft = max(8192, 1 << int(np.ceil(np.log2(len(x) * 4))))
    widmo = np.abs(np.fft.rfft(x, nfft)) ** 2
    f = np.fft.rfftfreq(nfft, 1.0 / fs)
    pas = (f >= lo) & (f <= hi)
    if not pas.any():
        return float("nan"), float("-inf")
    fp = f[pas][np.argmax(widmo[pas])]
    sygn = (np.abs(f - fp) <= 0.1) | (np.abs(f - 2 * fp) <= 0.2)
    snr = 10 * np.log10(widmo[pas & sygn].sum() / (widmo[pas & ~sygn].sum() + 1e-12) + 1e-12)
    return float(fp * 60.0), float(snr)


def tetno_rppg(t: np.ndarray, rgb: np.ndarray, fs: float = 30.0) -> tuple[float, float]:
    rgb = uzupelnij_nan(rgb)
    if not np.isfinite(rgb).all() or len(t) < fs * 4:
        return float("nan"), float("-inf")
    x = przeprobkuj(np.asarray(t, np.float64), rgb, fs)
    return tetno_z_sygnalu(pasmowy(pos_bvp(x, fs), fs), fs)


# --------------------------------------------------------------------------- audio
def normalizuj_audio(w: np.ndarray) -> np.ndarray:
    w = np.asarray(w, np.float32)
    return ((w - w.mean()) / np.sqrt(w.var() + 1e-7)).astype(np.float32)


def dopasuj_dlugosc(w: np.ndarray, n: int = AUDIO_SEG, start: int | None = None) -> np.ndarray:
    """Przytnij/dopelnij do n probek (domyslnie srodek)."""
    if len(w) >= n:
        s = (len(w) - n) // 2 if start is None else start
        return w[s:s + n]
    out = np.zeros(n, np.float32)
    s = (n - len(w)) // 2 if start is None else start
    out[s:s + len(w)] = w
    return out

In [ ]:
sys.path.insert(0, '/content')
import importlib, cechy as C
importlib.reload(C)

LANDMARKER = MODELE / 'face_landmarker.task'
if not LANDMARKER.exists():
    urllib.request.urlretrieve(C.FACE_LANDMARKER_URL, LANDMARKER)
print('FaceLandmarker:', LANDMARKER, LANDMARKER.stat().st_size // 1024, 'KB')

def rozpakuj(zip_path, cel):
    # Rozpakowuje zip raz (znacznik .ok).
    cel = pathlib.Path(cel); znacznik = cel / f'.ok_{pathlib.Path(zip_path).stem}'
    if not znacznik.exists():
        print('Rozpakowuję', zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(cel)
        znacznik.touch()
    return cel

def pobierz(url, cel):
    cel = pathlib.Path(cel)
    if not cel.exists():
        print('Pobieram', url)
        tmp = cel.with_suffix('.part')
        urllib.request.urlretrieve(url, tmp)
        tmp.rename(cel)
    return cel

### Narzędzia treningowe (augmentacja na GPU, sieć ResNet18 dla obrazów w skali szarości, eksport ONNX)

In [ ]:
import torchvision
from sklearn.metrics import f1_score, confusion_matrix, classification_report

class GrayNet(nn.Module):
    # Wejście: (B,1,H,W) w zakresie 0..1. Normalizacja i skalowanie są wewnątrz, więc w ONNX też.
    def __init__(self, n_klas, rozmiar_wew=None):
        super().__init__()
        m = torchvision.models.resnet18(weights='IMAGENET1K_V1')
        m.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(512, n_klas))
        self.m, self.rozmiar_wew = m, rozmiar_wew
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        if self.rozmiar_wew:
            x = F.interpolate(x, size=(self.rozmiar_wew, self.rozmiar_wew), mode='bilinear', align_corners=False)
        x = x.repeat(1, 3, 1, 1)
        return self.m((x - self.mean) / self.std)

def augmentuj(x, rot=15, skala=(0.9, 1.1), przes=0.08, flip=True):
    B, dev = x.shape[0], x.device
    u = lambda a, b, *s: torch.empty(B, *s, device=dev).uniform_(a, b)
    ang = u(-1, 1) * math.radians(rot); sc = u(*skala)
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    tx, ty = u(-przes, przes), u(-przes, przes)
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    if flip:
        theta[:, :, 0] *= torch.where(torch.rand(B, device=dev) < 0.5, -1.0, 1.0).unsqueeze(1)
    x = F.grid_sample(x, F.affine_grid(theta, x.shape, align_corners=False), padding_mode='border', align_corners=False)
    m = x.mean((2, 3), keepdim=True)
    x = (x - m) * u(0.6, 1.4, 1, 1, 1) + m + u(-0.15, 0.15, 1, 1, 1)
    x = x.clamp(1e-4, 1) ** torch.exp(u(-0.4, 0.4, 1, 1, 1))
    x = x + torch.randn_like(x) * u(0, 0.04, 1, 1, 1)
    return x.clamp(0, 1)

def trenuj_obrazy(model, Xtr, ytr, Xva, yva, epoki, lr, bs, wagi=None, nazwa='model'):
    # X*: uint8 tensory (N,1,H,W) na GPU, y*: long tensory na GPU.
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    kroki = epoki * math.ceil(len(Xtr) / bs)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=kroki, pct_start=0.15)
    crit = nn.CrossEntropyLoss(weight=wagi, label_smoothing=0.1)
    najlepszy, stan = -1, None
    for ep in range(epoki):
        model.train(); t0 = time.time(); suma = 0
        perm = torch.randperm(len(Xtr), device=DEVICE)
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]
            x = augmentuj(Xtr[idx].float() / 255)
            with torch.autocast('cuda', dtype=torch.bfloat16):
                loss = crit(model(x), ytr[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
            suma += loss.item() * len(idx)
        pred = przewiduj(model, Xva, bs * 2)
        f1 = f1_score(yva.cpu(), pred, average='macro'); acc = (pred == yva.cpu().numpy()).mean()
        print(f'[{nazwa}] epoka {ep+1}/{epoki} loss={suma/len(Xtr):.4f} val_acc={acc:.4f} val_F1={f1:.4f} ({time.time()-t0:.0f}s)')
        if f1 > najlepszy:
            najlepszy, stan = f1, {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(stan)
    print(f'[{nazwa}] najlepsze macro-F1 = {najlepszy:.4f}')
    return model

@torch.no_grad()
def przewiduj(model, X, bs=1024):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        with torch.autocast('cuda', dtype=torch.bfloat16):
            out.append(model(X[i:i + bs].float() / 255).argmax(1).cpu())
    return torch.cat(out).numpy()

def eksport_onnx(model, dummy, sciezka):
    import onnxruntime as ort
    model = model.eval().float().cpu(); dummy = dummy.cpu()
    kw = dict(input_names=['input'], output_names=['logits'], opset_version=17)
    try:
        torch.onnx.export(model, (dummy,), str(sciezka), dynamo=False, **kw)
    except Exception as e:
        print('Eksporter klasyczny niedostępny, używam dynamo:', type(e).__name__)
        torch.onnx.export(model, (dummy,), str(sciezka), dynamo=True, external_data=False, **kw)
    so = ort.SessionOptions(); so.intra_op_num_threads = 4
    s = ort.InferenceSession(str(sciezka), so, providers=['CPUExecutionProvider'])
    out = s.run(None, {s.get_inputs()[0].name: dummy.numpy()})[0]
    with torch.no_grad():
        ref = model(dummy); ref = getattr(ref, 'logits', ref).numpy()
    print(f'Zapisano {sciezka} ({pathlib.Path(sciezka).stat().st_size/1e6:.1f} MB), max różnica ONNX vs PyTorch: {np.abs(out-ref).max():.2e}')

## 1. Stan oczu: MRL Eye Dataset
~84 tys. zdjęć oczu (IR i światło widzialne). Etykieta jest w nazwie pliku (5. pole: 0 = zamknięte, 1 = otwarte).
Podział na trening/walidację **po osobach**, żeby model nie „zapamiętał” konkretnych ludzi.
Jeśli serwer uczelni nie działa, pobierz `mrlEyes_2018_01.zip` (np. mirror z Kaggle) i wrzuć do `dane/`.

In [ ]:
SCIEZKA_OCZY = MODELE / 'oczy.onnx'
if SCIEZKA_OCZY.exists():
    print('oczy.onnx już istnieje, pomijam (usuń plik, aby trenować ponownie).')
else:
    mrl_zip = DANE / 'mrlEyes_2018_01.zip'
    if not mrl_zip.exists() and not list(DANE.glob('*mrl*')):
        try:
            pobierz('http://mrl.cs.vsb.cz/data/eyedataset/mrlEyes_2018_01.zip', mrl_zip)
        except Exception as e:
            print('Nie udało się pobrać automatycznie:', e)
    for z in DANE.glob('*.zip'):
        if 'mrl' in z.name.lower():
            rozpakuj(z, LOKALNE / 'mrl')
    pliki = [p for p in (LOKALNE / 'mrl').rglob('*') if p.suffix.lower() in ('.png', '.jpg', '.jpeg')]
    print('Obrazów:', len(pliki))
    wzor = re.compile(r's(\d{4})_\d+_\d_\d_(\d)_\d_\d_\d+$')
    X, y, osoba = [], [], []
    for p in pliki:
        m = wzor.match(p.stem)
        if m:
            zamk, os_ = int(m.group(2) == '0'), int(m.group(1))
        else:  # mirror z folderami Open/Closed
            nazwa = p.parent.name.lower()
            if 'clos' in nazwa: zamk = 1
            elif 'open' in nazwa: zamk = 0
            else: continue
            os_ = hash(p.stem[:5]) % 1000
        img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if img is None: continue
        X.append(cv2.resize(img, (C.ROZMIAR_OKA, C.ROZMIAR_OKA), interpolation=cv2.INTER_AREA)); y.append(zamk); osoba.append(os_)
    X, y, osoba = np.stack(X), np.array(y), np.array(osoba)
    osoby = np.unique(osoba); val_os = osoby[::5]
    va = np.isin(osoba, val_os)
    print(f'Trening {(~va).sum()}, walidacja {va.sum()} ({len(val_os)} osób); zamknięte: {y.mean():.2%}')
    T = lambda a: torch.from_numpy(a).to(DEVICE)
    Xtr, Xva = T(X[~va][:, None]), T(X[va][:, None]); ytr, yva = T(y[~va]).long(), T(y[va]).long()
    model_oczu = trenuj_obrazy(GrayNet(2), Xtr, ytr, Xva, yva, epoki=6, lr=1e-3, bs=512, nazwa='oczy')
    print(classification_report(yva.cpu(), przewiduj(model_oczu, Xva), target_names=['otwarte', 'zamknięte'], digits=3))
    eksport_onnx(model_oczu, torch.zeros(1, 1, C.ROZMIAR_OKA, C.ROZMIAR_OKA), SCIEZKA_OCZY)
    del X, Xtr, Xva; torch.cuda.empty_cache()

## 2. Emocje z twarzy: FER-2013
Pobieranie przez `kagglehub`. Jeśli poprosi o logowanie: na kaggle.com → Settings → *Create New Token*,
a potem w Colabie (ikona klucza, **Secrets**) dodaj `KAGGLE_USERNAME` i `KAGGLE_KEY`.
Spodziewana dokładność to ok. 68–72% (tyle osiągają dobre modele na tym zbiorze; ludzie mają ok. 65%).

In [ ]:
SCIEZKA_TWARZ = MODELE / 'emocje_twarz.onnx'
if SCIEZKA_TWARZ.exists():
    print('emocje_twarz.onnx już istnieje, pomijam.')
else:
    try:
        from google.colab import userdata
        for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY'):
            try: os.environ[k] = userdata.get(k)
            except Exception: pass
    except Exception: pass
    import kagglehub
    fer = pathlib.Path(kagglehub.dataset_download('msambare/fer2013'))
    def wczytaj(folder):
        X, y = [], []
        for k, nazwa in enumerate(C.EMOCJE_TWARZ):
            for p in sorted((folder / nazwa).glob('*')):
                img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    X.append(cv2.resize(img, (C.ROZMIAR_TWARZY, C.ROZMIAR_TWARZY))); y.append(k)
        return np.stack(X)[:, None], np.array(y)
    Xtr, ytr = wczytaj(fer / 'train'); Xva, yva = wczytaj(fer / 'test')
    print('Trening', Xtr.shape, 'test', Xva.shape, 'liczność klas:', np.bincount(ytr))
    wagi = torch.tensor((len(ytr) / np.bincount(ytr)) ** 0.5, dtype=torch.float32, device=DEVICE); wagi /= wagi.mean()
    T = lambda a: torch.from_numpy(a).to(DEVICE)
    Xtr_t, Xva_t, ytr_t, yva_t = T(Xtr), T(Xva), T(ytr).long(), T(yva).long()
    model_tw = trenuj_obrazy(GrayNet(len(C.EMOCJE_TWARZ), rozmiar_wew=112), Xtr_t, ytr_t, Xva_t, yva_t,
                             epoki=30, lr=1e-3, bs=256, wagi=wagi, nazwa='twarz')
    p = przewiduj(model_tw, Xva_t)
    print(classification_report(yva, p, target_names=C.EMOCJE_TWARZ, digits=3))
    print(confusion_matrix(yva, p))
    eksport_onnx(model_tw, torch.zeros(1, 1, C.ROZMIAR_TWARZY, C.ROZMIAR_TWARZY), SCIEZKA_TWARZ)
    del Xtr_t, Xva_t; torch.cuda.empty_cache()

## 3. Emocje z głosu: RAVDESS (wav2vec2)
Pobiera `Audio_Speech_Actors_01-24.zip` (~200 MB) z Zenodo. Dotrenowujemy `facebook/wav2vec2-base`
na 4-sekundowych fragmentach 16 kHz. Podział **po aktorach**: 1–20 trening, 21–24 walidacja.
Uwaga: RAVDESS to mowa *aktorska* po angielsku, więc na naturalnej polskiej mowie wyniki będą słabsze.

In [ ]:
SCIEZKA_GLOS = MODELE / 'emocje_glos.onnx'
if SCIEZKA_GLOS.exists():
    print('emocje_glos.onnx już istnieje, pomijam.')
else:
    import librosa
    from transformers import AutoModelForAudioClassification, get_cosine_schedule_with_warmup
    zip_r = pobierz('https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1',
                    DANE / 'Audio_Speech_Actors_01-24.zip')
    rozpakuj(zip_r, LOKALNE / 'ravdess')
    fale, ety, aktor = [], [], []
    for p in sorted((LOKALNE / 'ravdess').rglob('*.wav')):
        cz = p.stem.split('-')
        if len(cz) != 7: continue
        w, _ = librosa.load(p, sr=C.AUDIO_SR, mono=True)
        w, _ = librosa.effects.trim(w, top_db=30)
        fale.append(w.astype(np.float32)); ety.append(int(cz[2]) - 1); aktor.append(int(cz[6]))
    ety, aktor = np.array(ety), np.array(aktor)
    tr_idx, va_idx = np.flatnonzero(aktor <= 20), np.flatnonzero(aktor > 20)
    print('Nagrań:', len(fale), '| trening', len(tr_idx), '| walidacja', len(va_idx))

    class GlosDS(torch.utils.data.Dataset):
        def __init__(self, idx, trening): self.idx, self.tr = idx, trening
        def __len__(self): return len(self.idx)
        def __getitem__(self, i):
            j = self.idx[i]; w = fale[j]; n = C.AUDIO_SEG
            if self.tr:
                start = np.random.randint(0, abs(len(w) - n) + 1)
                w = C.dopasuj_dlugosc(w, n, start) * np.random.uniform(0.3, 1.5)
                w = w + np.random.randn(n).astype(np.float32) * np.random.uniform(0.0005, 0.01)
            else:
                w = C.dopasuj_dlugosc(w, n) + np.random.RandomState(j).randn(n).astype(np.float32) * 0.002
            return torch.from_numpy(C.normalizuj_audio(w)), int(ety[j])

    dl_tr = torch.utils.data.DataLoader(GlosDS(tr_idx, True), batch_size=16, shuffle=True, num_workers=2, drop_last=True)
    dl_va = torch.utils.data.DataLoader(GlosDS(va_idx, False), batch_size=32, num_workers=2)
    model_g = AutoModelForAudioClassification.from_pretrained(
        'facebook/wav2vec2-base', num_labels=len(C.EMOCJE_GLOS), attn_implementation='eager',
        id2label=dict(enumerate(C.EMOCJE_GLOS)), label2id={n: i for i, n in enumerate(C.EMOCJE_GLOS)}).to(DEVICE)
    model_g.freeze_feature_encoder()
    EPOKI = 25
    opt = torch.optim.AdamW([p for p in model_g.parameters() if p.requires_grad], lr=5e-5, weight_decay=1e-2)
    sched = get_cosine_schedule_with_warmup(opt, len(dl_tr) * 2, len(dl_tr) * EPOKI)
    najlepszy, stan = -1, None
    for ep in range(EPOKI):
        model_g.train(); t0 = time.time()
        for x, yb in dl_tr:
            x, yb = x.to(DEVICE), yb.to(DEVICE)
            with torch.autocast('cuda', dtype=torch.bfloat16):
                loss = F.cross_entropy(model_g(input_values=x).logits, yb, label_smoothing=0.1)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model_g.parameters(), 1.0); opt.step(); sched.step()
        model_g.eval(); pr, gt = [], []
        with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
            for x, yb in dl_va:
                pr.append(model_g(input_values=x.to(DEVICE)).logits.argmax(1).cpu()); gt.append(yb)
        pr, gt = torch.cat(pr).numpy(), torch.cat(gt).numpy()
        f1 = f1_score(gt, pr, average='macro')
        print(f'[głos] epoka {ep+1}/{EPOKI} loss={loss.item():.3f} val_acc={(pr==gt).mean():.3f} val_F1={f1:.3f} ({time.time()-t0:.0f}s)')
        if f1 > najlepszy:
            najlepszy, stan = f1, {k: v.detach().cpu().clone() for k, v in model_g.state_dict().items()}
            pr_best = pr
    model_g.load_state_dict(stan)
    print('Najlepsze macro-F1:', round(najlepszy, 4))
    print(classification_report(gt, pr_best, target_names=C.EMOCJE_GLOS, digits=3))

    class GlosONNX(nn.Module):
        def __init__(self, m): super().__init__(); self.m = m
        def forward(self, x): return self.m(input_values=x).logits
    eksport_onnx(GlosONNX(model_g), torch.randn(1, C.AUDIO_SEG), SCIEZKA_GLOS)
    del model_g; torch.cuda.empty_cache()

## 4. Próg ziewania: YawDD (kalibracja)
Pobierz YawDD z IEEE DataPort (darmowe konto) i wrzuć do `dane/YawDD/` (zip lub rozpakowane).
Używamy nagrań z folderu *Mirror*, gdzie nazwy plików kończą się na `-Normal`, `-Talking` lub `-Yawning`.
Próg MAR (otwarcie ust) ustawiamy tak, żeby normalna mowa prawie nigdy go nie przekraczała.

In [ ]:
YAW = DANE / 'YawDD'
if 'MAR_YAWN' in KONFIG:
    print('Próg już skalibrowany:', KONFIG['MAR_YAWN'])
elif not YAW.exists():
    KONFIG.setdefault('MAR_YAWN', 0.5); zapisz_konfig()
    print('Brak dane/YawDD: używam domyślnego progu MAR = 0.5')
else:
    for z in YAW.rglob('*.zip'): rozpakuj(z, LOKALNE / 'yawdd')
    filmy = [p for r in (YAW, LOKALNE / 'yawdd') if r.exists() for p in r.rglob('*')
             if p.suffix.lower() in ('.avi', '.mp4') and re.search(r'(normal|talking|yawning)', p.stem, re.I)]
    print('Filmów z etykietą:', len(filmy))
    from concurrent.futures import ProcessPoolExecutor
    import multiprocessing as mproc
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mproc.get_context('spawn')) as ex:
        mary = list(ex.map(C.mar_z_filmu, map(str, filmy), [str(LANDMARKER)] * len(filmy)))
    ziew = [m for p, m in zip(filmy, mary) if 'yawn' in p.stem.lower() and len(m)]
    norm = np.concatenate([m for p, m in zip(filmy, mary) if 'yawn' not in p.stem.lower() and len(m)])
    prog = float(np.clip(np.percentile(norm, 99.5), 0.3, 0.9))
    wykryte = np.mean([(m > prog).sum() >= C.TRACKER_FPS for m in ziew])  # >=1 s powyżej progu
    print(f'Próg MAR_YAWN = {prog:.3f} | ziewnięcie wykryte w {wykryte:.0%} filmów "Yawning"')
    KONFIG['MAR_YAWN'] = prog; zapisz_konfig()

## 5. Poziom zmęczenia: UTA-RLDD
Pobierz UTA-RLDD (strona projektu, linki do części `Fold*_part*.zip`) i wrzuć zipy (albo rozpakowane foldery)
do `dane/UTA-RLDD/`. Struktura: `<osoba>/0.mov` (czujny), `5.mov` (obniżona czujność), `10.mov` (senny).
Całość to ok. 30 h wideo i **dużo** GB, więc na początek wystarczy 1–2 foldy.

Każdy film przechodzi przez **ten sam** pipeline co na laptopie (MediaPipe + sieć oczu → okna 60 s co 10 s
→ 15 cech: PERCLOS, mrugnięcia, mikrosen, ziewanie, ruchy głowy). Na tych cechach uczymy małą sieć MLP.
Cechy każdego filmu są zapisywane na Dysku, więc po przerwaniu sesji obliczenia wznawiają się.
Ocena: walidacja krzyżowa **po osobach** (GroupKFold). Uczciwie: rozróżnianie 3 poziomów u nowych osób jest trudne
(w literaturze ok. 60–65%), a „czujny vs senny” wychodzi wyraźnie lepiej.

In [ ]:
RLDD = DANE / 'UTA-RLDD'
CECHY_DIR = DANE / 'cechy_rldd'; CECHY_DIR.mkdir(exist_ok=True)
SCIEZKA_ZM = MODELE / 'zmeczenie.onnx'
MAKS_OSOB = None   # np. 20, żeby szybciej przetestować
ETYK = {'0': 0, '5': 1, '10': 2}

filmy = []
if RLDD.exists():
    for z in sorted(RLDD.rglob('*.zip')): rozpakuj(z, LOKALNE / 'rldd')
    for r in (RLDD, LOKALNE / 'rldd'):
        if r.exists():
            filmy += [p for p in r.rglob('*') if p.suffix.lower() in ('.mov', '.mp4', '.avi', '.m4v') and p.stem in ETYK]
osoby = sorted({p.parent.name for p in filmy})[:MAKS_OSOB]
filmy = [p for p in filmy if p.parent.name in osoby]
print(f'Filmów: {len(filmy)}, osób: {len(osoby)}' if filmy else 'Brak danych UTA-RLDD w dane/UTA-RLDD: sekcja pominięta.')

if filmy:
    from concurrent.futures import ProcessPoolExecutor, as_completed
    import multiprocessing as mproc
    prog_mar = KONFIG.get('MAR_YAWN', 0.5)
    oczy = str(SCIEZKA_OCZY) if SCIEZKA_OCZY.exists() else None
    if oczy is None: print('UWAGA: brak oczy.onnx, uruchom najpierw sekcję 1 (inaczej cechy będą z blendshape).')
    zadania = [(p, CECHY_DIR / f'{p.parent.name}_{p.stem}.npy') for p in filmy]
    zadania = [(p, o) for p, o in zadania if not o.exists()]
    print('Do przetworzenia:', len(zadania), 'filmów (pozostałe w cache)')
    t0 = time.time()
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mproc.get_context('spawn')) as ex:
        fut = {ex.submit(C.cechy_z_filmu, str(p), str(LANDMARKER), oczy, prog_mar): (p, o) for p, o in zadania}
        for k, f in enumerate(as_completed(fut), 1):
            p, o = fut[f]
            try:
                np.save(o, f.result())
            except Exception as e:
                print('Błąd', p, e)
            print(f'\r{k}/{len(zadania)} ({(time.time()-t0)/60:.1f} min)', end='')
    print()

In [ ]:
if filmy and not SCIEZKA_ZM.exists():
    from sklearn.model_selection import GroupKFold
    X, y, g = [], [], []
    for p in filmy:
        o = CECHY_DIR / f'{p.parent.name}_{p.stem}.npy'
        if o.exists():
            a = np.load(o); X.append(a); y += [ETYK[p.stem]] * len(a); g += [p.parent.name] * len(a)
    X, y, g = np.concatenate(X), np.array(y), np.array(g)
    print('Okien:', X.shape, 'klasy:', np.bincount(y))

    class ZmeczenieMLP(nn.Module):
        def __init__(self, mu, sd, n_we, n_kl=3):
            super().__init__()
            self.register_buffer('mu', torch.tensor(mu, dtype=torch.float32))
            self.register_buffer('sd', torch.tensor(sd, dtype=torch.float32))
            self.net = nn.Sequential(nn.Linear(n_we, 64), nn.GELU(), nn.Dropout(0.3),
                                     nn.Linear(64, 64), nn.GELU(), nn.Dropout(0.3), nn.Linear(64, n_kl))
        def forward(self, x):
            x = (x - self.mu) / self.sd
            x = torch.where(torch.isnan(x), torch.zeros_like(x), x).clamp(-6, 6)
            return self.net(x)

    def ucz_mlp(Xa, ya, epoki=150):
        mu, sd = np.nanmean(Xa, 0), np.nanstd(Xa, 0) + 1e-6
        m = ZmeczenieMLP(mu, sd, Xa.shape[1]).to(DEVICE)
        opt = torch.optim.AdamW(m.parameters(), lr=3e-3, weight_decay=1e-3)
        Xt, yt = torch.tensor(Xa, device=DEVICE), torch.tensor(ya, device=DEVICE)
        w = torch.tensor(len(ya) / np.bincount(ya, minlength=3).clip(1), dtype=torch.float32, device=DEVICE)
        for _ in range(epoki):
            m.train()
            for idx in torch.randperm(len(Xt), device=DEVICE).split(256):
                loss = F.cross_entropy(m(Xt[idx]), yt[idx], weight=w / w.mean(), label_smoothing=0.05)
                opt.zero_grad(); loss.backward(); opt.step()
        return m.eval()

    pr = np.zeros_like(y)
    for k, (tr, te) in enumerate(GroupKFold(n_splits=max(2, min(5, len(set(g))))).split(X, y, g)):
        m = ucz_mlp(X[tr], y[tr])
        with torch.no_grad(): pr[te] = m(torch.tensor(X[te], device=DEVICE)).argmax(1).cpu().numpy()
        print(f'fold {k+1}: acc={(pr[te]==y[te]).mean():.3f}')
    print(classification_report(y, pr, target_names=C.ZMECZENIE_KLASY, digits=3))
    print(confusion_matrix(y, pr))
    bin_ = y != 1
    print(f'Czujny vs senny (bez klasy środkowej): acc={(pr[bin_]==y[bin_]).mean():.3f}')

    m = ucz_mlp(X, y)
    eksport_onnx(m, torch.zeros(1, X.shape[1]), SCIEZKA_ZM)
    KONFIG['FEATURE_NAMES'] = C.FEATURE_NAMES; zapisz_konfig()
elif SCIEZKA_ZM.exists():
    print('zmeczenie.onnx już istnieje, pomijam.')

## 6. Tętno z kamery (rPPG): walidacja na UBFC-rPPG
Metoda POS nie wymaga treningu, a tu sprawdzamy, jak dokładnie działa z naszym pipeline'em.
Pobierz UBFC-rPPG **DATASET_2** (link do Google Drive na stronie zbioru) i wrzuć foldery `subject*`
(z plikami `vid.avi` i `ground_truth.txt`) do `dane/UBFC-rPPG/`.
Wynik: średni błąd bezwzględny (MAE) w uderzeniach na minutę dla okien 10 s.

In [ ]:
UBFC = DANE / 'UBFC-rPPG'
if not UBFC.exists() or not (any(UBFC.rglob('ground_truth.txt')) or any(UBFC.rglob('*.zip'))):
    print('Brak nagrań w dane/UBFC-rPPG: sekcja pominięta.')
else:
    for z in UBFC.rglob('*.zip'): rozpakuj(z, LOKALNE / 'ubfc')
    gts = [p for r in (UBFC, LOKALNE / 'ubfc') if r.exists() for p in r.rglob('ground_truth.txt')
           if (p.parent / 'vid.avi').exists()]
    print('Nagrań:', len(gts))
    from concurrent.futures import ProcessPoolExecutor
    import multiprocessing as mproc
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=mproc.get_context('spawn')) as ex:
        wyniki = list(ex.map(C.rgb_z_filmu, [str(p.parent / 'vid.avi') for p in gts], [str(LANDMARKER)] * len(gts)))
    bledy, pary = [], []
    for gtp, (t, rgb, fps) in zip(gts, wyniki):
        ppg = np.loadtxt(gtp)
        ppg = ppg[0] if ppg.ndim == 2 else ppg
        ppg = np.interp(np.linspace(0, 1, len(t)), np.linspace(0, 1, len(ppg)), ppg)
        for s in np.arange(0, t[-1] - 10, 2.0):
            w = (t >= s) & (t < s + 10)
            hr_pos, snr = C.tetno_rppg(t[w], rgb[w], 30.0)
            hr_gt, _ = C.tetno_z_sygnalu(C.pasmowy(ppg[w], fps), fps)
            if np.isfinite(hr_pos):
                pary.append((hr_gt, hr_pos, snr))
    pary = np.array(pary); e = np.abs(pary[:, 0] - pary[:, 1])
    dobre = pary[:, 2] > 0
    wynik = {'MAE_bpm': float(e.mean()), 'RMSE_bpm': float(np.sqrt((e**2).mean())),
             'Pearson_r': float(np.corrcoef(pary[:, 0], pary[:, 1])[0, 1]),
             'MAE_bpm_SNR>0dB': float(e[dobre].mean()) if dobre.any() else None,
             'odsetek_okien_SNR>0dB': float(dobre.mean()), 'okien': int(len(pary))}
    print(json.dumps(wynik, indent=2))
    (MODELE / 'rppg_wyniki.json').write_text(json.dumps(wynik, indent=2))

## 7. Gotowe: pobierz modele na laptopa
Skopiuj zawartość `MyDrive/MonitorZalogi/modele/` do folderu `laptop/modele/` (przez Dysk Google
albo pobierz zip z komórki poniżej).

In [ ]:
for p in sorted(MODELE.iterdir()):
    print(f'{p.name:28s} {p.stat().st_size/1e6:8.1f} MB')
shutil.make_archive('/content/modele', 'zip', MODELE)
from google.colab import files
files.download('/content/modele.zip')